In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)
import warnings
import os
import gc
import shutil
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        
    def forward(self, logits, labels):
        probs = F.softmax(logits, dim=-1)
        labels_one_hot = F.one_hot(labels, num_classes=logits.shape[-1]).float()
        pt = (probs * labels_one_hot).sum(dim=-1)  
        focal_weight = (1 - pt) ** self.gamma
        ce_loss = F.cross_entropy(logits, labels, reduction='none')
        focal_loss = focal_weight * ce_loss
        if self.alpha is not None:
            alpha_t = torch.tensor(self.alpha, device=logits.device)[labels]
            focal_loss = alpha_t * focal_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:
class FocalLossTrainer(Trainer):    
    def __init__(self, *args, focal_loss_fn=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss_fn = focal_loss_fn
        
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss = self.focal_loss_fn(logits, labels)        
        return (loss, outputs) if return_outputs else loss

In [ ]:
label2id = {
    "Clear Reply": 0,
    "Ambivalent": 1,
    "Clear Non-Reply": 2
}
id2label = {v: k for k, v in label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': accuracy, 'f1': f1}

In [ ]:
class EarlyStopping(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
from huggingface_hub import login

login()

In [ ]:
augmented_gpt_ds = load_dataset("gabrielstefan04/qevasion-gpt5.1-augmented")
train_df = augmented_gpt_ds['train'].to_pandas()
val_df = augmented_gpt_ds['validation'].to_pandas()

In [ ]:
class_counts = train_df['clarity_label'].value_counts()
total_samples = len(train_df)

alpha_values = {}
for label in clarity_labels:
    count = class_counts.get(label, 0)
    alpha_values[label] = total_samples / (len(clarity_labels) * count)

alpha_list = [alpha_values[label] for label in clarity_labels]

In [ ]:
MODERNBERT_MODEL = "answerdotai/ModernBERT-large"
MODERNBERT_MAX_LENGTH = 3000

MODERNBERT_PARAMS = {
    "learning_rate": 2.02197598390604e-05,
    "weight_decay": 0.09129256917296077,
    "warmup_ratio": 0.04304687629932645,
    "gradient_accumulation_steps": 1,
}

In [ ]:
modernbert_tokenizer = AutoTokenizer.from_pretrained(MODERNBERT_MODEL)
data_collator = DataCollatorWithPadding(tokenizer=modernbert_tokenizer)

In [ ]:
def prepare_datasets(train_df, val_df, tokenizer, max_length):    
    def tokenize_function(examples):
        inputs = [
            f"Question: {q}\nAnswer: {a}"
            for q, a in zip(examples["question"], examples["interview_answer"])
        ]
        tokenized = tokenizer(
            inputs,
            truncation=True,
            max_length=max_length,
            padding=False
        )
        tokenized["labels"] = [label2id[l] for l in examples["clarity_label"]]
        return tokenized
    
    train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
    val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
    
    train_tokenized = train_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=train_dataset.column_names
    )
    val_tokenized = val_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=val_dataset.column_names
    )
    
    return train_tokenized, val_tokenized

original_ds = load_dataset("ailsntua/QEvasion")
test_df = original_ds['test'].to_pandas()
train_tokenized, val_tokenized = prepare_datasets(
    train_df, val_df,
    modernbert_tokenizer,
    MODERNBERT_MAX_LENGTH
)

In [ ]:
def evaluate_on_test(trainer, tokenizer, test_df, max_length, model_name, experiment_name):
    def tokenize_test(examples):
        texts = [
            f"Question: {q}\nAnswer: {a}"
            for q, a in zip(examples['question'], examples['interview_answer'])
        ]
        return tokenizer(texts, truncation=True, padding=False, max_length=max_length)
    
    test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
    test_tokenized = test_dataset.map(
        tokenize_test,
        batched=True,
        remove_columns=[col for col in test_dataset.column_names if col not in ['clarity_label']]
    )
    
    predictions_output = trainer.predict(test_tokenized)
    predictions = np.argmax(predictions_output.predictions, axis=-1)
    predicted_labels = [id2label[pred] for pred in predictions]
    
    y_true = test_df['clarity_label'].tolist()
    y_pred = predicted_labels
    
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', labels=clarity_labels, zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=clarity_labels, zero_division=0)
    
    per_class_f1 = f1_score(y_true, y_pred, average=None, labels=clarity_labels, zero_division=0)
    
    print(f"\nAccuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    
    print("\nPer-class F1 scores:")
    for label, f1_val in zip(clarity_labels, per_class_f1):
        print(f"  {label}: {f1_val:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, labels=clarity_labels, zero_division=0))
    
    print("Confusion Matrix:")
    cm = confusion_matrix(y_true, y_pred, labels=clarity_labels)
    print(f"Labels: {clarity_labels}")
    print(cm)
    
    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'per_class_f1': {label: f1_val for label, f1_val in zip(clarity_labels, per_class_f1)}
    }

### Weighted Cross Entropy

In [ ]:
class_weights = torch.tensor(alpha_list, dtype=torch.float32).to(device)

model_baseline = AutoModelForSequenceClassification.from_pretrained(
    MODERNBERT_MODEL,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)
model_baseline.config.use_cache = False

class WeightedCETrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

training_args_baseline = TrainingArguments(
    output_dir="./results_weighted_ce",
    
    learning_rate=MODERNBERT_PARAMS["learning_rate"],
    weight_decay=MODERNBERT_PARAMS["weight_decay"],
    warmup_ratio=MODERNBERT_PARAMS["warmup_ratio"],
    gradient_accumulation_steps=MODERNBERT_PARAMS["gradient_accumulation_steps"],
    
    num_train_epochs=15,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    logging_steps=50,
    logging_strategy="steps",
    report_to="none",
    
    fp16=False,
    bf16=True,
    optim="adamw_torch",
    seed=SEED,
)

trainer_baseline = WeightedCETrainer(
    model=model_baseline,
    args=training_args_baseline,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=modernbert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStopping(patience=3, metric_name="eval_f1", greater_is_better=True)],
    class_weights=class_weights,
)

trainer_baseline.train()

In [ ]:
results_baseline = evaluate_on_test(
    trainer_baseline,
    modernbert_tokenizer,
    test_df,
    MODERNBERT_MAX_LENGTH,
    "ModernBERT-large",
    "Weighted Cross Entropy"
)

In [ ]:
del model_baseline, trainer_baseline
torch.cuda.empty_cache()
gc.collect()
if os.path.exists("./results_weighted_ce"):
    shutil.rmtree("./results_weighted_ce")


### Focal Loss with gamma = 2.0

In [ ]:
focal_loss_fn = FocalLoss(alpha=alpha_list, gamma=2.0, reduction='mean')

In [ ]:
model_focal = AutoModelForSequenceClassification.from_pretrained(
    MODERNBERT_MODEL,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)
model_focal.config.use_cache = False

training_args_focal = TrainingArguments(
    output_dir="./results_focal_gamma1",
    
    learning_rate=MODERNBERT_PARAMS["learning_rate"],
    weight_decay=MODERNBERT_PARAMS["weight_decay"],
    warmup_ratio=MODERNBERT_PARAMS["warmup_ratio"],
    gradient_accumulation_steps=MODERNBERT_PARAMS["gradient_accumulation_steps"],
 
    num_train_epochs=15,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    logging_steps=50,
    logging_strategy="steps",
    report_to="none",
    
    fp16=False,
    bf16=True,
    optim="adamw_torch",
    seed=SEED,
)

trainer_focal = FocalLossTrainer(
    model=model_focal,
    args=training_args_focal,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=modernbert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStopping(patience=3, metric_name="eval_f1", greater_is_better=True)],
    focal_loss_fn=focal_loss_fn,
)

trainer_focal.train()

In [ ]:
results_focal = evaluate_on_test(
    trainer_focal,
    modernbert_tokenizer,
    test_df,
    MODERNBERT_MAX_LENGTH,
    "ModernBERT-large",
    "Focal Loss (gamma=2.0)"
)

In [ ]:
del model_focal, trainer_focal, focal_loss_fn
torch.cuda.empty_cache()
gc.collect()
if os.path.exists("./results_focal_gamma1"):
    shutil.rmtree("./results_focal_gamma1")

### Results Summary

In [ ]:
all_results = [
    {"Method": "Weighted Cross Entropy", **results_baseline},
    {"Method": "Focal Loss (γ=1.0)", **results_focal},
]

results_df = pd.DataFrame([
    {
        "Method": r["Method"],
        "Accuracy": r["accuracy"],
        "Macro F1": r["macro_f1"],
        "Weighted F1": r["weighted_f1"]
    }
    for r in all_results
])

print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
per_class_df = pd.DataFrame([
    {
        "Method": r["Method"],
        "Clear Reply": r["per_class_f1"]["Clear Reply"],
        "Ambivalent": r["per_class_f1"]["Ambivalent"],
        "Clear Non-Reply": r["per_class_f1"]["Clear Non-Reply"]
    }
    for r in all_results
])

print(per_class_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))